<a href="https://colab.research.google.com/github/irajamuller/error_corrections/blob/main/UA3_UA4_Estabilizadores_Shor_Code_Ruido_Medida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit qiskit_aer pylatexenc --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.0 MB/s eta 0:00:00


In [2]:
# Classes do qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator, Aer
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, array_to_latex
from qiskit_aer.noise import (
    NoiseModel,
    depolarizing_error,
    ReadoutError,
)
import numpy as np

In [ ]:
# ─────────────────────────────────────────────
# PARÂMETROS DE RUÍDO (ajuste à vontade)
# ─────────────────────────────────────────────
MEAS_FLIP_PROB   = 0.05   # P(bit-flip no resultado de medição de síndrome)
ANCILLA_DEPOL    = 0.01   # erro depolarizante nas ancillas (reset + porta 1Q)
NUM_ROUNDS       = 5      # rodadas de medição de síndrome; votação majoritária


# ─────────────────────────────────────────────
# MODELO DE RUÍDO (Aer)
# ─────────────────────────────────────────────

def build_noise_model(
    meas_flip: float = MEAS_FLIP_PROB,
    depol_1q:  float = ANCILLA_DEPOL,
) -> NoiseModel:
    """
    Ruído focado nas ancillas de síndrome:
      • ReadoutError nas medições de ancilla
      • Depolarizante 1Q em h/x gates das ancillas
    """
    nm = NoiseModel()

    # Erro de medição (bit-flip simétrico)
    p0g1 = meas_flip          # mede 0, era 1
    p1g0 = meas_flip          # mede 1, era 0
    ro_err = ReadoutError([[1 - p1g0, p1g0],
                           [p0g1,     1 - p0g1]])
    # aplica em TODOS os qubits (ancillas incluídas)
    nm.add_all_qubit_readout_error(ro_err)

    # Depolarizante nas portas de 1 qubit usadas nas ancillas
    dep1 = depolarizing_error(depol_1q, 1)
    nm.add_all_qubit_quantum_error(dep1, ['h', 'x', 'reset'])

    return nm


# ─────────────────────────────────────────────
# FUNÇÕES DO CIRCUITO (idênticas ao original,
# mas sem usar ClassicalRegisters internos —
# serão passados por rodada)
# ─────────────────────────────────────────────

def encode(qc: QuantumCircuit, d):
    qc.barrier(label='Encode')
    qc.cx(d[0], d[3])
    qc.cx(d[0], d[6])
    qc.h(d[0]); qc.h(d[3]); qc.h(d[6])
    for i in [0, 3, 6]:
        qc.cx(d[i], d[i + 1])
        qc.cx(d[i], d[i + 2])


def logical_x_transversal(qc: QuantumCircuit, d):
    qc.barrier(label='Transversal X_L')
    for i in range(9):
        qc.x(d[i])


def inject_error(qc: QuantumCircuit, d, errors):
    qc.barrier(label='Error')
    if not errors:
        return
    for qubit, etype in errors:
        if etype in ('X', 'Y'):
            qc.x(d[qubit])
        if etype in ('Z', 'Y'):
            qc.z(d[qubit])


def _measure_z1_z6_round(
    qc: QuantumCircuit,
    d, a,
    s0: ClassicalRegister,
    s1: ClassicalRegister,
    s2: ClassicalRegister,
):
    """Uma rodada de medição dos estabilizadores Z1-Z6 (bit-flip)."""
    for i in range(6):
        qc.reset(a[i])

    # Bloco 0
    qc.h(a[0]); qc.cz(d[0], a[0]); qc.cz(d[1], a[0]); qc.h(a[0])
    qc.h(a[1]); qc.cz(d[1], a[1]); qc.cz(d[2], a[1]); qc.h(a[1])
    # Bloco 1
    qc.h(a[2]); qc.cz(d[3], a[2]); qc.cz(d[4], a[2]); qc.h(a[2])
    qc.h(a[3]); qc.cz(d[4], a[3]); qc.cz(d[5], a[3]); qc.h(a[3])
    # Bloco 2
    qc.h(a[4]); qc.cz(d[6], a[4]); qc.cz(d[7], a[4]); qc.h(a[4])
    qc.h(a[5]); qc.cz(d[7], a[5]); qc.cz(d[8], a[5]); qc.h(a[5])

    qc.measure(a[0], s0[0]); qc.measure(a[1], s0[1])
    qc.measure(a[2], s1[0]); qc.measure(a[3], s1[1])
    qc.measure(a[4], s2[0]); qc.measure(a[5], s2[1])


def _measure_z7_z8_round(
    qc: QuantumCircuit,
    d, a,
    sz: ClassicalRegister,
):
    """Uma rodada de medição dos estabilizadores Z7-Z8 (phase-flip)."""
    qc.reset(a[0]); qc.reset(a[1])

    qc.h(a[0])
    for i in [0, 1, 2, 3, 4, 5]:
        qc.h(d[i]); qc.cz(a[0], d[i]); qc.h(d[i])
    qc.h(a[0])
    qc.measure(a[0], sz[0])

    qc.h(a[1])
    for i in [3, 4, 5, 6, 7, 8]:
        qc.h(d[i]); qc.cz(a[1], d[i]); qc.h(d[i])
    qc.h(a[1])
    qc.measure(a[1], sz[1])


# ─────────────────────────────────────────────
# VOTAÇÃO MAJORITÁRIA (pós-processamento)
# ─────────────────────────────────────────────

def majority_vote_syndromes(
    counts: dict[str, int],
    num_rounds: int,
    num_shots: int,
) -> dict:
    """
    Para cada shot:
      - Extrai as síndromes de cada rodada
      - Aplica votação majoritária bit a bit entre rodadas
      - Retorna dicionário {síndrome_votada: contagem}

    Layout dos bits no counts-string do Qiskit (da direita para esquerda):
      logical_bit | sz_r(R-1)..sz_r0 | s2_r(R-1)..s2_r0 |
      s1_r(R-1)..s1_r0 | s0_r(R-1)..s0_r0
    Cada registro tem 2 bits; são NUM_ROUNDS registros por tipo.
    """
    voted: dict[str, int] = {}

    for bitstring, cnt in counts.items():
        bits = bitstring.replace(' ', '')[::-1]  # LSB primeiro

        # Posições (LSB first):
        # logical_bit: bits[0]          → 1 bit
        # s0 rodadas:  bits[1 .. 2*R]   → 2 bits × R rodadas
        # s1 rodadas:  bits[2R+1..4R]
        # s2 rodadas:  bits[4R+1..6R]
        # sz rodadas:  bits[6R+1..8R]

        R = num_rounds
        logical = bits[0]

        def extract_rounds(start, nbits=2):
            """Retorna lista de inteiros, uma por rodada."""
            result = []
            for r in range(R):
                chunk = bits[start + r * nbits: start + r * nbits + nbits]
                result.append(int(chunk[::-1], 2))   # volta para MSB-first
            return result

        s0_rounds = extract_rounds(1,       2)
        s1_rounds = extract_rounds(1 + 2*R, 2)
        s2_rounds = extract_rounds(1 + 4*R, 2)
        sz_rounds = extract_rounds(1 + 6*R, 2)

        def vote(rounds_vals: list[int], nbits: int) -> int:
            """Votação majoritária bit a bit."""
            result = 0
            for b in range(nbits):
                ones = sum((v >> b) & 1 for v in rounds_vals)
                if ones > len(rounds_vals) / 2:
                    result |= (1 << b)
            return result

        s0_v = vote(s0_rounds, 2)
        s1_v = vote(s1_rounds, 2)
        s2_v = vote(s2_rounds, 2)
        sz_v = vote(sz_rounds, 2)

        key = f"s0={s0_v:02b} s1={s1_v:02b} s2={s2_v:02b} sz={sz_v:02b} log={logical}"
        voted[key] = voted.get(key, 0) + cnt

    return voted


# ─────────────────────────────────────────────
# CONSTRUÇÃO DO CIRCUITO COMPLETO
# ─────────────────────────────────────────────

def create_circuit(
    psi: str = '0',
    errors = None,
    num_rounds: int = NUM_ROUNDS,
) -> QuantumCircuit:
    """
    Cria o circuito do código de Shor com:
      - `num_rounds` rodadas de medição de síndrome (sem correção dinâmica
        entre rodadas — a correção é feita por votação majoritária em
        pós-processamento, pois o Qiskit 2.x não suporta lógica clássica
        acumulativa de forma nativa sem mid-circuit feed-forward complexo).
      - Correção final baseada nos bits de síndrome da última rodada
        (compatível com if_test do Qiskit 2.x).
    """
    if errors is None:
        errors = []

    R = num_rounds

    q_code     = QuantumRegister(9,  'code')
    q_anc_x    = QuantumRegister(6,  'anc_x')
    q_anc_z    = QuantumRegister(2,  'anc_z')

    # Um par de registros clássicos por rodada
    c_s0 = [ClassicalRegister(2, f's0_r{r}') for r in range(R)]
    c_s1 = [ClassicalRegister(2, f's1_r{r}') for r in range(R)]
    c_s2 = [ClassicalRegister(2, f's2_r{r}') for r in range(R)]
    c_sz = [ClassicalRegister(2, f'sz_r{r}') for r in range(R)]

    c_logical  = ClassicalRegister(1, 'logical')

    qc = QuantumCircuit(
        q_code, q_anc_x, q_anc_z,
        *c_s0, *c_s1, *c_s2, *c_sz,
        c_logical,
    )

    # ── Estado inicial ──────────────────────────────
    qc.barrier(label='Initial state')
    if psi != '0':
        qc.x(q_code[0])

    # ── Encode ──────────────────────────────────────
    encode(qc, q_code)

    # ── Operação lógica ─────────────────────────────
    logical_x_transversal(qc, q_code)

    # ── Injeção de erro ─────────────────────────────
    inject_error(qc, q_code, errors)

    # ── Múltiplas rodadas de síndrome ───────────────
    for r in range(R):
        qc.barrier(label=f'Syndrome round {r}')
        _measure_z1_z6_round(qc, q_code, q_anc_x,
                              c_s0[r], c_s1[r], c_s2[r])
        _measure_z7_z8_round(qc, q_code, q_anc_z, c_sz[r])

    # ── Correção baseada na ÚLTIMA rodada ───────────
    # (a votação majoritária completa é feita em pós-processamento)
    # Usamos a última rodada como proxy; o pós-proc. corrige erros de medição.
    last = R - 1
    qc.barrier(label='Correction X (last round)')
    with qc.if_test((c_s0[last], 0b01)): qc.x(q_code[0])
    with qc.if_test((c_s0[last], 0b11)): qc.x(q_code[1])
    with qc.if_test((c_s0[last], 0b10)): qc.x(q_code[2])
    with qc.if_test((c_s1[last], 0b01)): qc.x(q_code[3])
    with qc.if_test((c_s1[last], 0b11)): qc.x(q_code[4])
    with qc.if_test((c_s1[last], 0b10)): qc.x(q_code[5])
    with qc.if_test((c_s2[last], 0b01)): qc.x(q_code[6])
    with qc.if_test((c_s2[last], 0b11)): qc.x(q_code[7])
    with qc.if_test((c_s2[last], 0b10)): qc.x(q_code[8])

    qc.barrier(label='Correction Z (last round)')
    with qc.if_test((c_sz[last], 0b01)): qc.z(q_code[0])
    with qc.if_test((c_sz[last], 0b11)): qc.z(q_code[3])
    with qc.if_test((c_sz[last], 0b10)): qc.z(q_code[6])

    # ── Decode ──────────────────────────────────────
    qc.barrier(label='Decode')
    for i in [0, 3, 6]:
        qc.cx(q_code[i], q_code[i + 1])
        qc.cx(q_code[i], q_code[i + 2])
    qc.h(q_code[0]); qc.h(q_code[3]); qc.h(q_code[6])
    qc.cx(q_code[0], q_code[3])
    qc.cx(q_code[0], q_code[6])
    qc.ccx(q_code[3], q_code[6], q_code[0])

    # ── Medição final ────────────────────────────────
    qc.barrier(label='Fim')
    qc.measure(q_code[0], c_logical[0])

    return qc


# ─────────────────────────────────────────────
# SIMULAÇÃO
# ─────────────────────────────────────────────

def run_simulation(
    psi:        str   = '0',
    errors            = None,
    num_rounds: int   = NUM_ROUNDS,
    shots:      int   = 2048,
    meas_flip:  float = MEAS_FLIP_PROB,
    depol_1q:   float = ANCILLA_DEPOL,
    seed:       int   = 42,
) -> dict:
    """
    Executa o circuito com ruído e retorna:
      - 'raw_counts'   : contagens brutas do simulador
      - 'voted_counts' : contagens após votação majoritária das síndromes
      - 'logical_0'    : fração de shots com bit lógico = 0
      - 'logical_1'    : fração de shots com bit lógico = 1
      - 'circuit'      : o QuantumCircuit criado
    """
    if errors is None:
        errors = []

    qc = create_circuit(psi=psi, errors=errors, num_rounds=num_rounds)

    noise_model = build_noise_model(meas_flip=meas_flip, depol_1q=depol_1q)

    backend = AerSimulator(noise_model=noise_model, seed_simulator=seed)
    tqc     = transpile(qc, backend, optimization_level=0)
    job     = backend.run(tqc, shots=shots)
    result  = job.result()
    raw     = result.get_counts()

    voted   = majority_vote_syndromes(raw, num_rounds=num_rounds, num_shots=shots)

    # Estatística do bit lógico após votação
    log0 = log1 = 0
    for key, cnt in voted.items():
        if key.endswith('log=0'):
            log0 += cnt
        else:
            log1 += cnt

    return {
        'raw_counts':   raw,
        'voted_counts': voted,
        'logical_0':    log0 / shots,
        'logical_1':    log1 / shots,
        'circuit':      qc,
    }


# ─────────────────────────────────────────────
# DEMO
# ─────────────────────────────────────────────

if __name__ == '__main__':
    print("=" * 60)
    print("  Código de Shor — Ruído + Votação Majoritária (Qiskit 2.x)")
    print("=" * 60)

    cenarios = [
        ("Sem erro",             '0', []),
        ("Erro X no qubit 4",    '0', [(4, 'X')]),
        ("Erro Z no qubit 0",    '0', [(0, 'Z')]),
        ("Erro Y no qubit 7",    '0', [(7, 'Y')]),
        ("Estado |1>, sem erro", '1', []),
        ("Estado |1>, erro X0",  '1', [(0, 'X')]),
    ]

    for descricao, psi, erros in cenarios:
        res = run_simulation(
            psi=psi,
            errors=erros,
            num_rounds=NUM_ROUNDS,
            shots=2048,
            meas_flip=MEAS_FLIP_PROB,
            depol_1q=ANCILLA_DEPOL,
        )
        esperado = psi  # X_L inverte, decode desfaz → deve voltar ao |psi>
        print(f"\n[{descricao}]  |psi>={psi}  esperado=|{esperado}>")
        print(f"  P(lógico=0) = {res['logical_0']:.3f}")
        print(f"  P(lógico=1) = {res['logical_1']:.3f}")
        top = sorted(res['voted_counts'].items(), key=lambda x: -x[1])[:3]
        print("  Top síndromes votadas:")
        for k, v in top:
            print(f"    {k}  →  {v} shots")

    # Exibe o circuito de um caso simples
    print("\n\nCircuito (sem erro, 2 rodadas para compacidade):")
    qc_demo = create_circuit(psi='0', errors=[], num_rounds=2)
    print(qc_demo.draw(output='text', fold=120))


In [ ]:
qc_demo = create_circuit(psi='0', errors=[(0, 'X')], num_rounds=2)
qc.draw('mpl', fold=-1)